In [1]:
pip install psycopg2-binary sqlalchemy

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   -------------------------------- ------- 2.1/2.6 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 9.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [19]:
import psycopg2

conn = psycopg2.connect(
    host="aws-1-ap-south-1.pooler.supabase.com",
    database="postgres",
    user="postgres.dijluzdqrykhbxglwltk",   # full Supabase username
    password="@@@@@Abcd@1234",              # your actual password
    port=6543                               # ✅ correct port for Supabase
)

cur = conn.cursor()
print("✅ Connected to Supabase PostgreSQL!")


✅ Connected to Supabase PostgreSQL!


In [20]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS users (
        user_id SERIAL PRIMARY KEY,
        user_name VARCHAR(100),
        phone VARCHAR(20),
        test_taken JSONB
    );
""")
conn.commit()
print("✅ Users table created!")


✅ Users table created!


In [21]:
data = [
    (1, "John Doe", "9876543210", '{"AQ-10": "Moderate", "ASRS": "Severe", "PHQ-9": "Moderate"}'),
    (2, "Alice Smith", "8765432109", None),
    (3, "Bob Johnson", "7654321098", '{"AUDIT": "Moderate", "DAST": "Moderate"}'),
    (4, "Clara Lee", "6543210987", '{"MSI-BPD": "Moderate", "EAT-26": "Severe"}'),
    (5, "David Kim", "5432109876", '{"AQ-10": "Severe", "GAD-7": "Moderate"}'),
    (6, "Emily Davis", "4321098765", '{"PHQ-9": "Severe", "PCL-5": "Moderate", "Mood Disorder Questionnaire": "Severe"}'),
    (7, "Frank Miller", "3210987654", '{"AUDIT": "Severe", "DAST": "Moderate", "Y-BOCS": "Moderate"}'),
    (8, "Grace Wilson", "2109876543", '{"ASRS": "Moderate", "MSI-BPD": "Severe"}'),
    (9, "Henry Brown", "1098765432", '{"EAT-26": "Moderate", "Mood Disorder Questionnaire": "Severe"}'),
    (10, "Irene Clark", "9988776655", '{"GAD-7": "Severe", "AQ-10": "Moderate", "PHQ-9": "Severe"}'),
]

for row in data:
    cur.execute("""
        INSERT INTO users (user_id, user_name, phone, test_taken)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (user_id) DO NOTHING;
    """, row)

conn.commit()
print("✅ Sample data inserted!")


✅ Sample data inserted!


In [22]:
cur.execute("SELECT * FROM users;")
rows = cur.fetchall()
for row in rows:
    print(row)


(1, 'John Doe', '9876543210', {'ASRS': 'Severe', 'AQ-10': 'Moderate', 'PHQ-9': 'Moderate'})
(2, 'Alice Smith', '8765432109', None)
(3, 'Bob Johnson', '7654321098', {'DAST': 'Moderate', 'AUDIT': 'Moderate'})
(4, 'Clara Lee', '6543210987', {'EAT-26': 'Severe', 'MSI-BPD': 'Moderate'})
(5, 'David Kim', '5432109876', {'AQ-10': 'Severe', 'GAD-7': 'Moderate'})
(6, 'Emily Davis', '4321098765', {'PCL-5': 'Moderate', 'PHQ-9': 'Severe', 'Mood Disorder Questionnaire': 'Severe'})
(7, 'Frank Miller', '3210987654', {'DAST': 'Moderate', 'AUDIT': 'Severe', 'Y-BOCS': 'Moderate'})
(8, 'Grace Wilson', '2109876543', {'ASRS': 'Moderate', 'MSI-BPD': 'Severe'})
(9, 'Henry Brown', '1098765432', {'EAT-26': 'Moderate', 'Mood Disorder Questionnaire': 'Severe'})
(10, 'Irene Clark', '9988776655', {'AQ-10': 'Moderate', 'GAD-7': 'Severe', 'PHQ-9': 'Severe'})


In [13]:
import pandas as pd
from sqlalchemy import create_engine

# Step 1: Load CSV
appointments_df = pd.read_csv(r"C:\Users\aamreen_quantum-i\OneDrive\Desktop\Symptoms_checker\symptoms_checker\CSV Data\doctor_appointment_load.csv")

# Step 2: Connection string (modify with your DB credentials)
USER = "postgres"       # e.g., postgres
PASSWORD = "12345678"   # e.g., 1234
HOST = "localhost"           # or your DB server IP
PORT = "5432"                # default postgres port
DBNAME = "agentdata"       # e.g., hospital_db

# Step 3: Create engine
engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")

# Step 4: Save DataFrame to Postgres (inside schema BHC)
appointments_df.to_sql(
    "appointments_df",  # table name in postgres
    engine,
    schema="BHC",           # schema name you created
    if_exists="replace",    # options: 'fail', 'replace', 'append'
    index=False             # don’t save pandas index
)


print("Data saved successfully to Postgres!")


Data saved successfully to Postgres!


In [6]:
pip install psycopg2-binary sqlalchemy

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------- ----------- 1.8/2.6 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 7.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [22]:
import pandas as pd
from sqlalchemy import create_engine
from fastapi import FastAPI

app = FastAPI()

# Function to fetch table data from Postgres
@app.post("/fetch_table_from_postgres")
def fetch_table_from_postgres(table_name: str):
    """
    Connects to PostgreSQL and fetches the specified table into a pandas DataFrame.
    Args:
        table_name (str): The name of the table in Postgres.
    Returns:
        pd.DataFrame: Table data.
    """
    
    schema="BHC"                     
    user="postgres"                    
    password="12345678"            
    host="localhost"
    port=5432
    database="agentdata"   
    
    # Create connection string
    connection_uri = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"

    # Create engine
    engine = create_engine(connection_uri)

    # SQL query
    query = f'SELECT * FROM "{schema}".{table_name}'

    # Read into DataFrame
    df = pd.read_sql(query, engine)

    return df.to_dict(orient="records")

In [20]:
pip install fastapi

Note: you may need to restart the kernel to use updated packages.
